In [21]:
# ============================================
# CELDA 1: IMPORTS
# ============================================

import pandas as pd
import numpy as np

from pathlib import Path

from nfstream import NFStreamer

import warnings

warnings.filterwarnings("ignore")

In [22]:
# ============================================
# CELDA 2: CONFIGURACIÓN DE DIRECTORIOS
# ============================================

BASE_DIR = Path.cwd()

NETWORK_DIR = BASE_DIR / "network_data"

TELEMETRY_DIR = BASE_DIR / "telemetry_data"

OUTPUT_DIR = BASE_DIR / "output_phase1"

OUTPUT_DIR.mkdir(exist_ok=True)

# ============================================
# OUTPUT FILES
# ============================================

FLOW_OUTPUT = OUTPUT_DIR / "raw_flow_features.csv"

LOG_OUTPUT = OUTPUT_DIR / "raw_log_features.csv"

WINDOW_OUTPUT = OUTPUT_DIR / "sliding_window_features.csv"

print("OUTPUT DIR:")

print(OUTPUT_DIR)

OUTPUT DIR:
e:\Proyectos VS code\ProcesamientoData\output_phase1


In [23]:
# ============================================
# CELDA 3: BÚSQUEDA DE ARCHIVOS
# ============================================

pcap_files = list(

    NETWORK_DIR.rglob("*.pcap")

)

log_files = list(

    TELEMETRY_DIR.rglob("*.log")

)

print("PCAPS ENCONTRADOS:")

print(len(pcap_files))

print()

print("LOGS ENCONTRADOS:")

print(len(log_files))

PCAPS ENCONTRADOS:
33

LOGS ENCONTRADOS:
70


In [24]:
# ============================================
# CELDA 4: LIMPIAR CSVs PREVIOS
# ============================================

if FLOW_OUTPUT.exists():

    FLOW_OUTPUT.unlink()

if LOG_OUTPUT.exists():

    LOG_OUTPUT.unlink()

if WINDOW_OUTPUT.exists():

    WINDOW_OUTPUT.unlink()

print("CSV PREVIOS ELIMINADOS")

CSV PREVIOS ELIMINADOS


In [25]:
# ============================================
# CELDA 5: EXTRACCIÓN DE FEATURES IT (PCAPS)
# ============================================

pcap_counter = 0

for pcap_file in pcap_files:

    pcap_counter += 1

    print()

    print(f"PROCESANDO PCAP {pcap_counter}")

    print(pcap_file.name)

    try:

        streamer = NFStreamer(

            source=str(pcap_file),
            statistical_analysis=True

        )

        flow_records = []

        for flow in streamer:

            flow_records.append({

                "timestamp":

                    flow.bidirectional_first_seen_ms,

                "src_ip":

                    flow.src_ip,

                "src_port":

                    flow.src_port,

                "dst_ip":

                    flow.dst_ip,

                "dst_port":

                    flow.dst_port,

                "protocol":

                    flow.protocol,

                "duration_ms":

                    flow.bidirectional_duration_ms,

                "bidirectional_packets":

                    flow.bidirectional_packets,

                "bidirectional_bytes":

                    flow.bidirectional_bytes,

                "src2dst_packets":

                    flow.src2dst_packets,

                "dst2src_packets":

                    flow.dst2src_packets,

                "src2dst_bytes":

                    flow.src2dst_bytes,

                "dst2src_bytes":

                    flow.dst2src_bytes,

                "src2dst_duration_ms":

                    flow.src2dst_duration_ms,

                "dst2src_duration_ms":

                    flow.dst2src_duration_ms,

                "application_name":

                    flow.application_name,

                "application_category":

                    flow.application_category_name,

                "attack_type":

                    pcap_file.parent.name,

                # ============================================
                # LABEL CORREGIDO
                # ============================================

                "traffic_label":

                    (

                        "NORMAL"

                        if pcap_file.parent.name == "normal_pcaps"

                        else "ATTACK"

                    )

            })

        flow_df = pd.DataFrame(flow_records)

        # ============================================
        # EXPORTACIÓN STREAMING
        # ============================================

        if pcap_counter == 1:

            flow_df.to_csv(

                FLOW_OUTPUT,
                index=False,
                mode="w"

            )

        else:

            flow_df.to_csv(

                FLOW_OUTPUT,
                index=False,
                mode="a",
                header=False

            )

        print(

            f"FLOWS EXTRAÍDOS: {len(flow_df)}"

        )

        del flow_df
        del flow_records

    except Exception as e:

        print("ERROR:")

        print(e)


PROCESANDO PCAP 1
normal_1.pcap
FLOWS EXTRAÍDOS: 3634

PROCESANDO PCAP 2
normal_10.pcap
FLOWS EXTRAÍDOS: 3078

PROCESANDO PCAP 3
normal_11.pcap
FLOWS EXTRAÍDOS: 3076

PROCESANDO PCAP 4
normal_12.pcap
FLOWS EXTRAÍDOS: 4074

PROCESANDO PCAP 5
normal_13.pcap
FLOWS EXTRAÍDOS: 6111

PROCESANDO PCAP 6
normal_2.pcap
FLOWS EXTRAÍDOS: 3471

PROCESANDO PCAP 7
normal_3.pcap
FLOWS EXTRAÍDOS: 2965

PROCESANDO PCAP 8
normal_4.pcap
FLOWS EXTRAÍDOS: 5970

PROCESANDO PCAP 9
normal_5.pcap
FLOWS EXTRAÍDOS: 6954

PROCESANDO PCAP 10
normal_6.pcap
FLOWS EXTRAÍDOS: 8101

PROCESANDO PCAP 11
normal_7.pcap
FLOWS EXTRAÍDOS: 5404

PROCESANDO PCAP 12
normal_8.pcap
FLOWS EXTRAÍDOS: 3418

PROCESANDO PCAP 13
normal_9.pcap
FLOWS EXTRAÍDOS: 3643

PROCESANDO PCAP 14
normal_IoT_2.pcap
FLOWS EXTRAÍDOS: 12496

PROCESANDO PCAP 15
normal_IoT_3.pcap
FLOWS EXTRAÍDOS: 13093

PROCESANDO PCAP 16
injection_normal1.pcap
FLOWS EXTRAÍDOS: 150166

PROCESANDO PCAP 17
injection_normal2.pcap
FLOWS EXTRAÍDOS: 159906

PROCESANDO PCAP 18
i

In [26]:
# ============================================
# CELDA 6: EXTRACCIÓN DE FEATURES OT (LOGS)
# ============================================

log_counter = 0

for log_file in log_files:

    log_counter += 1

    print()

    print(f"PROCESANDO LOG {log_counter}")

    print(log_file.name)

    try:

        lines = []

        with open(

            log_file,
            "r",
            encoding="utf-8",
            errors="ignore"

        ) as file:

            for line_number, line in enumerate(file):

                lines.append({

                    "line_number":

                        line_number,

                    "raw_log":

                        line.strip(),

                    "log_length":

                        len(line),

                    "device_type":

                        log_file.stem,

                    "attack_type":

                        log_file.parent.name,

                    # ============================================
                    # LABEL CORREGIDO
                    # ============================================

                    "traffic_label":

                        (

                            "NORMAL"

                            if log_file.parent.name == "IoT_original_normal"

                            else "ATTACK"

                        )

                })

        log_df = pd.DataFrame(lines)

        # ============================================
        # EXPORTACIÓN STREAMING
        # ============================================

        if log_counter == 1:

            log_df.to_csv(

                LOG_OUTPUT,
                index=False,
                mode="w"

            )

        else:

            log_df.to_csv(

                LOG_OUTPUT,
                index=False,
                mode="a",
                header=False

            )

        print(

            f"LOGS EXTRAÍDOS: {len(log_df)}"

        )

        del log_df
        del lines

    except Exception as e:

        print("ERROR:")

        print(e)


PROCESANDO LOG 1
IoT_normal_fridge_1.log
LOGS EXTRAÍDOS: 40057

PROCESANDO LOG 2
IoT_normal_fridge_2.log
LOGS EXTRAÍDOS: 190162

PROCESANDO LOG 3
IoT_normal_fridge_3.log
LOGS EXTRAÍDOS: 153150

PROCESANDO LOG 4
IoT_normal_Garage_door_1.log
LOGS EXTRAÍDOS: 56948

PROCESANDO LOG 5
IoT_normal_Garage_door_2.log
LOGS EXTRAÍDOS: 188978

PROCESANDO LOG 6
IoT_normal_Garage_door_3.log
LOGS EXTRAÍDOS: 124216

PROCESANDO LOG 7
IoT_normal_GPS_Tracker_1.log
LOGS EXTRAÍDOS: 48492

PROCESANDO LOG 8
IoT_normal_GPS_Tracker_2.log
LOGS EXTRAÍDOS: 188979

PROCESANDO LOG 9
IoT_normal_GPS_Tracker_3.log
LOGS EXTRAÍDOS: 153147

PROCESANDO LOG 10
IoT_normal_modbus_1.log
LOGS EXTRAÍDOS: 48805

PROCESANDO LOG 11
IoT_normal_modbus_2.log
LOGS EXTRAÍDOS: 12901

PROCESANDO LOG 12
IoT_normal_modbus_3.log
LOGS EXTRAÍDOS: 165694

PROCESANDO LOG 13
IoT_normal_Motion_Light_1.log
LOGS EXTRAÍDOS: 45138

PROCESANDO LOG 14
IoT_normal_Motion_Light_2.log
LOGS EXTRAÍDOS: 147745

PROCESANDO LOG 15
IoT_normal_Motion_Light_3.log


In [27]:
# ============================================
# CELDA 7: GENERACIÓN DE VENTANAS DESLIZANTES
# ============================================

CHUNK_SIZE = 100000

WINDOW_SECONDS = 1

chunk_counter = 0

flow_reader = pd.read_csv(

    FLOW_OUTPUT,
    chunksize=CHUNK_SIZE,
    low_memory=True

)

for chunk in flow_reader:

    chunk_counter += 1

    print()

    print(f"PROCESANDO CHUNK {chunk_counter}")

    chunk["timestamp"] = pd.to_datetime(

        chunk["timestamp"],
        unit="ms",
        errors="coerce"

    )

    chunk = chunk.dropna(

        subset=["timestamp"]

    )

    chunk = chunk.sort_values(

        by="timestamp"

    )

    # ============================================
    # INTER-ARRIVAL TIME
    # ============================================

    chunk["inter_arrival_time"] = (

        chunk["timestamp"]
        .diff()
        .dt.total_seconds()

    )

    # ============================================
    # JITTER
    # ============================================

    chunk["jitter"] = (

        chunk["inter_arrival_time"]
        .diff()
        .abs()

    )

    # ============================================
    # VENTANAS DESLIZANTES
    # ============================================

    window_df = (

        chunk
        .set_index("timestamp")
        .resample(f"{WINDOW_SECONDS}s")
        .agg({

            "bidirectional_packets": "sum",

            "bidirectional_bytes": "sum",

            "inter_arrival_time": "mean",

            "jitter": "mean"

        })

        .reset_index()

    )

    window_df.rename(columns={

        "bidirectional_packets":

            "packet_count",

        "bidirectional_bytes":

            "byte_count"

    }, inplace=True)

    # ============================================
    # EXPORTACIÓN STREAMING
    # ============================================

    if chunk_counter == 1:

        window_df.to_csv(

            WINDOW_OUTPUT,
            index=False,
            mode="w"

        )

    else:

        window_df.to_csv(

            WINDOW_OUTPUT,
            index=False,
            mode="a",
            header=False

        )

    print(

        f"VENTANAS GENERADAS: {len(window_df)}"

    )

    del chunk
    del window_df


PROCESANDO CHUNK 1
VENTANAS GENERADAS: 1975011

PROCESANDO CHUNK 2
VENTANAS GENERADAS: 4427

PROCESANDO CHUNK 3
VENTANAS GENERADAS: 5472

PROCESANDO CHUNK 4
VENTANAS GENERADAS: 4475

PROCESANDO CHUNK 5
VENTANAS GENERADAS: 4507

PROCESANDO CHUNK 6
VENTANAS GENERADAS: 365043

PROCESANDO CHUNK 7
VENTANAS GENERADAS: 327404

PROCESANDO CHUNK 8
VENTANAS GENERADAS: 45577

PROCESANDO CHUNK 9
VENTANAS GENERADAS: 1584

PROCESANDO CHUNK 10
VENTANAS GENERADAS: 1934

PROCESANDO CHUNK 11
VENTANAS GENERADAS: 1981

PROCESANDO CHUNK 12
VENTANAS GENERADAS: 1578

PROCESANDO CHUNK 13
VENTANAS GENERADAS: 2219

PROCESANDO CHUNK 14
VENTANAS GENERADAS: 2153

PROCESANDO CHUNK 15
VENTANAS GENERADAS: 1577

PROCESANDO CHUNK 16
VENTANAS GENERADAS: 1745

PROCESANDO CHUNK 17
VENTANAS GENERADAS: 2923

PROCESANDO CHUNK 18
VENTANAS GENERADAS: 3026

PROCESANDO CHUNK 19
VENTANAS GENERADAS: 58977

PROCESANDO CHUNK 20
VENTANAS GENERADAS: 1862

PROCESANDO CHUNK 21
VENTANAS GENERADAS: 1952

PROCESANDO CHUNK 22
VENTANAS GENE

In [28]:
# ============================================
# CELDA 8: VALIDACIÓN FINAL DE OUTPUTS
# ============================================

print("===================================")
print("FASE 1 COMPLETADA")
print("===================================")

print()

print("OUTPUTS GENERADOS:")

print()

print("1.", FLOW_OUTPUT)

print("2.", LOG_OUTPUT)

print("3.", WINDOW_OUTPUT)

FASE 1 COMPLETADA

OUTPUTS GENERADOS:

1. e:\Proyectos VS code\ProcesamientoData\output_phase1\raw_flow_features.csv
2. e:\Proyectos VS code\ProcesamientoData\output_phase1\raw_log_features.csv
3. e:\Proyectos VS code\ProcesamientoData\output_phase1\sliding_window_features.csv


In [29]:
# ============================================
# CELDA 9: VALIDACIÓN DE MUESTRAS
# ============================================

print("===================================")
print("FLOW SAMPLE")
print("===================================")

flow_sample = pd.read_csv(

    FLOW_OUTPUT,
    nrows=5

)

print(flow_sample.head())

print()

print("===================================")
print("LOG SAMPLE")
print("===================================")

log_sample = pd.read_csv(

    LOG_OUTPUT,
    nrows=5

)

print(log_sample.head())

FLOW SAMPLE
       timestamp                                  src_ip  src_port  \
0  1554220344295                           192.168.1.103         0   
1  1554220356962                             192.168.1.1     56504   
2  1554220345294                           192.168.1.103         0   
3  1554220384413  2405:6e00:10ce:2c00:20c:29ff:feee:e07a     21787   
4  1554220343061                             192.168.1.1         0   

            dst_ip  dst_port  protocol  duration_ms  bidirectional_packets  \
0      224.0.0.253         0         2            0                      1   
1  239.255.255.250      1900        17           99                      7   
2      224.0.0.251         0         2            0                      1   
3  2600:1401:2::f0        53        17           22                      2   
4        224.0.0.1         0         2            0                      1   

   bidirectional_bytes  src2dst_packets  dst2src_packets  src2dst_bytes  \
0                   62 